# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [3]:
# AGENT FUNCTION (GENERALIZED — works on arbitrary phrasing, not just fixed keywords)

import re

# --- Intent detection patterns (not tied to specific test queries) ---

# Matches an actual arithmetic expression anywhere in the text (numbers + operators),
# so "5 + 3 * 2" or "100 divided by 4" are caught even without the word "calculate"
MATH_EXPR_RE = re.compile(
    r"[-+]?\d[\d,\.]*\s*(?:%|[\+\-\*/\^]|\bplus\b|\bminus\b|\btimes\b|\bdivided by\b)\s*[-+]?\d",
    re.IGNORECASE
)
# Matches common ways of *asking* for a calculation, as a secondary signal
MATH_TRIGGER_RE = re.compile(
    r"\b(calculate|compute|solve|evaluate|what('?s| is)|sum of|product of|square root|percent(age)? of)\b",
    re.IGNORECASE
)
# Matches common ways of asking for keyword/key-term extraction
KEYWORD_TRIGGER_RE = re.compile(
    r"\b(keywords?|key\s?terms?|key\s?words?|main\s?topics?|important\s?terms?|extract.*(terms|words))\b",
    re.IGNORECASE
)


def _looks_like_math(query: str) -> bool:
    """True if the query contains an actual arithmetic pattern, OR a math-asking
    phrase (calculate/compute/etc.) together with at least one digit."""
    return bool(MATH_EXPR_RE.search(query)) or (
        bool(MATH_TRIGGER_RE.search(query)) and bool(re.search(r"\d", query))
    )


def _normalize_math_text(query: str) -> str:
    """Convert natural-language math phrasing into a plain arithmetic string."""
    q = query.replace(",", "")
    # "X% of Y" / "X percent of Y"  ->  (X/100)*Y
    q = re.sub(r"(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)", r"(\1/100)*\2", q, flags=re.I)
    q = re.sub(r"(\d+(?:\.\d+)?)\s*percent(?:age)?\s*of\s*(\d+(?:\.\d+)?)", r"(\1/100)*\2", q, flags=re.I)
    q = re.sub(r"\bplus\b", "+", q, flags=re.I)
    q = re.sub(r"\bminus\b", "-", q, flags=re.I)
    q = re.sub(r"\btimes\b", "*", q, flags=re.I)
    q = re.sub(r"\bdivided by\b", "/", q, flags=re.I)
    q = q.replace("^", "**")
    return q


def _extract_expression(query: str) -> str:
    """Pull the longest contiguous numeric/operator span out of the query."""
    q = _normalize_math_text(query)
    candidates = re.findall(r"[\d\.\+\-\*/\(\)%\s]{3,}", q)
    numeric_candidates = [c.strip() for c in candidates if re.search(r"\d", c) and re.search(r"[\+\-\*/]", c)]
    if numeric_candidates:
        return max(numeric_candidates, key=len)
    m = re.search(r"[-+]?\d+(\.\d+)?", q)
    return m.group(0) if m else ""


def agent(query: str):
    """
    Routes ANY incoming query to the correct tool using pattern-based intent
    detection (not hardcoded to specific test strings), then returns:
    {"type": "calculation / keywords / general / error", "result": ...}
    """
    if not isinstance(query, str) or not query.strip():
        return {"type": "error", "result": "Empty or invalid query."}

    try:
        # --- Route 1: Anything that looks like a math request ---
        if _looks_like_math(query):
            expression = _extract_expression(query)
            if not expression:
                return {"type": "error", "result": "No mathematical expression found in the query."}

            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                return {"type": "error", "result": f"Could not evaluate expression: '{expression}'"}

            return {"type": "calculation", "result": calc_result}

        # --- Route 2: Anything that asks for keywords/key terms/main topics ---
        elif KEYWORD_TRIGGER_RE.search(query):
            cleaned = KEYWORD_TRIGGER_RE.sub("", query)
            cleaned = re.sub(r"\b(from|in|of|extract)\b", "", cleaned, flags=re.I).strip(" :,-")
            text = cleaned if cleaned else query

            keywords = extract_keywords(text)
            if not keywords:
                return {"type": "error", "result": "No keywords could be extracted from the given text."}

            return {"type": "keywords", "result": keywords}

        # --- Route 3: Everything else falls back to a general response ---
        else:
            return {
                "type": "general",
                "result": f"I received your query: \"{query}\". No specific tool matched, so here is a general response."
            }

    except Exception as e:
        return {"type": "error", "result": f"An unexpected error occurred: {str(e)}"}


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [4]:
# Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['intelligence', 'industries', 'artificial', 'transforming']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'I received your query: "What is machine learning?". No specific tool matched, so here is a general response.'}
--------------------------------------------------


In [5]:
# Generalization check — queries NOT used to design the routing rules

unseen_queries = [
    "What is 15% of 240?",
    "5 + 3 * 2",
    "What's 100 divided by 4?",
    "12 plus 8 minus 3",
    "Find the key terms in: The stock market rallied after strong earnings reports",
    "give me the main topics of this paragraph about climate change and renewable energy",
    "Tell me a joke",
    "How are you today?",
    "",
    "7*7",
]

for q in unseen_queries:
    print("Query:", repr(q))
    print("Response:", agent(q))
    print("-" * 50)


Query: 'What is 15% of 240?'
Response: {'type': 'calculation', 'result': '36.0'}
--------------------------------------------------
Query: '5 + 3 * 2'
Response: {'type': 'calculation', 'result': '11'}
--------------------------------------------------
Query: "What's 100 divided by 4?"
Response: {'type': 'calculation', 'result': '25.0'}
--------------------------------------------------
Query: '12 plus 8 minus 3'
Response: {'type': 'calculation', 'result': '17'}
--------------------------------------------------
Query: 'Find the key terms in: The stock market rallied after strong earnings reports'
Response: {'type': 'keywords', 'result': ['market', 'after', 'earnings', 'stock', 'reports']}
--------------------------------------------------
Query: 'give me the main topics of this paragraph about climate change and renewable energy'
Response: {'type': 'keywords', 'result': ['energy', 'paragraph', 'renewable', 'about', 'change']}
--------------------------------------------------
Query: 'T

In [6]:
# Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): calculate 50*5
Response: {'type': 'calculation', 'result': '250'}
Enter query (type 'exit' to stop): exit


In [7]:
# BONUS: Logging + Extra Tool + Improved Routing
#
# This cell is optional and does not modify the required agent()
# implementation above — it just layers extra features on top,
# per the "Bonus" section of the assignment.

import datetime

# --- Bonus Tool: Text Summarizer (very simple, first-N-words) ---
def summarize_text(text: str, num_words: int = 10) -> str:
    """Return a short summary (first N words) of the given text."""
    try:
        words = text.split()
        summary = " ".join(words[:num_words])
        return summary + ("..." if len(words) > num_words else "")
    except Exception:
        return "Error in summarization"


# --- Simple in-memory log of every agent call ---
call_log = []

def logged_agent(query: str):
    """
    Wraps agent() with:
    - logging of every call (timestamp, query, response)
    - an extra 'summarize' route
    """
    timestamp = datetime.datetime.now().isoformat(timespec="seconds")
    query_lower = query.lower()

    if "summarize" in query_lower:
        match = re.search(r"summarize\s+(?:this|the following)?\s*[:\-]?\s*(.*)", query, re.IGNORECASE)
        text = match.group(1).strip() if match and match.group(1).strip() else query
        response = {"type": "summary", "result": summarize_text(text)}
    else:
        response = agent(query)

    call_log.append({
        "timestamp": timestamp,
        "query": query,
        "response": response
    })
    return response


# Quick bonus demo
bonus_queries = [
    "Summarize this: Artificial intelligence is transforming industries by automating tasks, improving decision making, and enabling new products",
    "Calculate 10 * (2 + 3)",
    "asdkjas invalid query ###"
]

for q in bonus_queries:
    print("Query:", q)
    print("Response:", logged_agent(q))
    print("-" * 50)

print("\nCall log entries:", len(call_log))


Query: Summarize this: Artificial intelligence is transforming industries by automating tasks, improving decision making, and enabling new products
Response: {'type': 'summary', 'result': 'Artificial intelligence is transforming industries by automating tasks, improving decision...'}
--------------------------------------------------
Query: Calculate 10 * (2 + 3)
Response: {'type': 'calculation', 'result': '50'}
--------------------------------------------------
Query: asdkjas invalid query ###
Response: {'type': 'general', 'result': 'I received your query: "asdkjas invalid query ###". No specific tool matched, so here is a general response.'}
--------------------------------------------------

Call log entries: 3


## Agentic Version (Free LLM — Groq)

The version above is rule-based (pattern/regex-driven, generalized to arbitrary phrasing). This section goes further: an actual LLM decides which tool to call by reasoning over the query, using native tool-use / function calling — the same mechanism real agent frameworks are built on.

This uses **Groq**, which has a genuinely free tier (no credit card required) and runs open models like Llama 3.3 very fast, via an OpenAI-compatible API.

**Get a free key:**
1. Go to [console.groq.com](https://console.groq.com) → sign in (Google/GitHub is fine)
2. **API Keys** → **Create API Key** → copy it (starts with `gsk_...`)
3. In Colab: click the 🔑 **Secrets** icon in the left sidebar → add secret named `GROQ_API_KEY` → paste the key → enable notebook access

**Without a key**, `agentic_agent()` automatically falls back to the rule-based `agent()` above, so the notebook still runs end-to-end.


In [8]:
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    pass  # not in Colab, or secret not set — agentic_agent() will fall back to agent()


In [9]:
# Safer calculator (AST-based, no eval — avoids arbitrary code execution)

import ast
import operator

_ALLOWED_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
    ast.FloorDiv: operator.floordiv,
}

def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("Unsupported or unsafe expression")

def safe_calculator(expression: str) -> str:
    """Evaluate a mathematical expression without using eval()."""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_eval_node(tree.body))
    except Exception:
        return "Error in calculation"


In [10]:
# AGENTIC AGENT — a free LLM (Groq / Llama 3.3) decides which tool to call

import json as _json

try:
    from openai import OpenAI  # Groq exposes an OpenAI-compatible API
    _GROQ_KEY = os.environ.get("GROQ_API_KEY")
    _client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=_GROQ_KEY) if _GROQ_KEY else None
    _HAS_KEY = _client is not None
except Exception:
    _HAS_KEY = False

# OpenAI-style function/tool schema (Groq uses this same format)
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression and return the numeric result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "A math expression, e.g. '20 + 5' or '(3*4)/2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_keywords",
            "description": "Extract the most relevant keywords from a piece of text.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "The text to extract keywords from."}
                },
                "required": ["text"]
            }
        }
    }
]

_TOOL_IMPL = {
    "calculator": lambda **kw: safe_calculator(kw["expression"]),
    "extract_keywords": lambda **kw: extract_keywords(kw["text"]),
}

SYSTEM_PROMPT = (
    "You are a task-routing agent. For every user query, decide whether it needs "
    "the calculator tool, the extract_keywords tool, or neither. "
    "If neither tool applies, just answer directly and briefly in plain text. "
    "Only call a tool when it is clearly the right fit for the query."
)


def agentic_agent(query: str) -> dict:
    """
    True agentic routing: a free LLM (Groq/Llama 3.3) reasons over ANY query and
    decides which tool (if any) to call via native tool-use — no hardcoded
    patterns. Falls back to the rule-based agent() if no key is configured.
    """
    if not isinstance(query, str) or not query.strip():
        return {"type": "error", "result": "Empty or invalid query."}

    if not _HAS_KEY:
        return agent(query)

    try:
        response = _client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": query},
            ],
            tools=TOOLS,
            tool_choice="auto",
        )

        message = response.choices[0].message
        tool_calls = message.tool_calls

        if not tool_calls:
            text = (message.content or "").strip()
            return {"type": "general", "result": text or "No response generated."}

        call = tool_calls[0]
        tool_name = call.function.name
        tool_args = _json.loads(call.function.arguments)
        tool_fn = _TOOL_IMPL.get(tool_name)

        if tool_fn is None:
            return {"type": "error", "result": f"Unknown tool requested: {tool_name}"}

        result = tool_fn(**tool_args)
        type_map = {"calculator": "calculation", "extract_keywords": "keywords"}
        result_type = type_map.get(tool_name, "general")

        if result == "Error in calculation":
            return {"type": "error", "result": f"Could not evaluate: {tool_args.get('expression')}"}

        return {"type": result_type, "result": result}

    except Exception as e:
        return {"type": "error", "result": f"Agentic agent failed: {str(e)}"}


In [11]:
# Test the agentic agent — same unseen queries as the rule-based generalization check

for q in unseen_queries:
    print("Query:", repr(q))
    print("Response:", agentic_agent(q))
    print("-" * 50)

print("\nRunning with GROQ_API_KEY set:", _HAS_KEY)


Query: 'What is 15% of 240?'
Response: {'type': 'error', 'result': 'Agentic agent failed: Error code: 400 - {\'error\': {\'message\': "Failed to call a function. Please adjust your prompt. See \'failed_generation\' for more details.", \'type\': \'invalid_request_error\', \'code\': \'tool_use_failed\', \'failed_generation\': \'<function=calculator({"expression": "0.15 * 240"})</function>\'}}'}
--------------------------------------------------
Query: '5 + 3 * 2'
Response: {'type': 'calculation', 'result': '11'}
--------------------------------------------------
Query: "What's 100 divided by 4?"
Response: {'type': 'calculation', 'result': '25.0'}
--------------------------------------------------
Query: '12 plus 8 minus 3'
Response: {'type': 'error', 'result': 'Agentic agent failed: Error code: 400 - {\'error\': {\'message\': "Failed to call a function. Please adjust your prompt. See \'failed_generation\' for more details.", \'type\': \'invalid_request_error\', \'code\': \'tool_use_faile

In [12]:
# Interactive Mode (Agentic — Groq/Llama 3.3 decides the routing)

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agentic_agent(user_input))


Enter query (type 'exit' to stop): calculate 0*0
Response: {'type': 'calculation', 'result': '0'}
Enter query (type 'exit' to stop): what is agentic ai?
Response: {'type': 'general', 'result': 'Agentic AI refers to artificial intelligence systems that are capable of autonomous decision-making and action, often with a sense of purpose, intention, or goal-directed behavior. These systems can perceive their environment, make decisions based on that perception, and take actions to achieve their objectives, much like humans or animals. Agentic AI combines aspects of machine learning, robotics, and cognitive architectures to create agents that can interact with and influence their surroundings.'}
Enter query (type 'exit' to stop): exit
